### News

In [15]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.3 MB/s eta 0:00:00
 

In [16]:
from unsloth import FastLanguageModel
import torch

print("Unsloth loaded")
print("CUDA available:", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth loaded
CUDA available: True


load qwen

In [17]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


total weights      : 3.257 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.166 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.760 GiB  free 11.192 GiB  reserve 11.166 GiB
  cuda:1  budget  13.00 GiB  weights  1.497 GiB  free 11.499 GiB  reserve 11.009 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory the quantiser keeps for its own load-time buffers)
note: tied embeddings: model.embed_tokens pinned with the head so accelerate does not have to mirror the weight across devices


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [29]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.2: Fast Qwen3 patching. Transformers: 4.54.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


In [8]:
!find /kaggle/input -maxdepth 3 -type f | head -50

In [32]:
from datasets import load_dataset

files = {
    "train": "/kaggle/input/datasets/utkarshsonawane2/training1/pyopt_train.jsonl",
    "validation": "/kaggle/input/datasets/utkarshsonawane2/training1/pyopt_val.jsonl",
    "test": "/kaggle/input/datasets/utkarshsonawane2/training1/pyopt_test.jsonl",
}

pyopt = load_dataset("json", data_files=files)

dataset = pyopt["train"]
val_dataset = pyopt["validation"]
test_dataset = pyopt["test"]

print("Train:", len(dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 160
Validation: 30
Test: 40


Let's see how row 100 looks like!

In [33]:
dataset[0]

{'id': 'string_concat-002',
 'category': 'string_concat',
 'fn': 'merge_words_2',
 'original': 'def merge_words_2(chunks):\n    out = ""\n    for piece in chunks:\n        out += piece\n    return out\n',
 'optimized': 'def merge_words_2(chunks):\n    return "".join(chunks)\n',
 'constraints': 'chunks is a list of strings.',
 'conversations': [{'role': 'user',
   'content': 'Optimize this Python function while preserving its behavior.\n\nInput constraints:\n- chunks is a list of strings.\n\nReturn only the optimized Python function.\n\n```python\ndef merge_words_2(chunks):\n    out = ""\n    for piece in chunks:\n        out += piece\n    return out\n```'},
  {'role': 'assistant',
   'content': '```python\ndef merge_words_2(chunks):\n    return "".join(chunks)\n```'}]}

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [34]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

example = test_dataset[0]
prompt = example["conversations"][0]["content"]

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False,
)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("CATEGORY:", example["category"])
print("\nPROMPT:\n", prompt)
print("\nQWEN ANSWER:\n", answer)
print("\nEXPECTED:\n", example["optimized"])

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CATEGORY: no_change

PROMPT:
 Optimize this Python function while preserving its behavior.

Input constraints:
- values is a list of integers. The function is already efficient and concise for this task; do not rewrite it unnecessarily.

Return only the optimized Python function.

```python
def map_squares_3(values):
    return [entry * entry for entry in values]
```

QWEN ANSWER:
 ```python
def map_squares_3(values):
    return [v ** 2 for v in values]
```

EXPECTED:
 def map_squares_3(values):
    return [entry * entry for entry in values]



In [36]:
import json
from pathlib import Path

baseline_results = []

for i, example in enumerate(test_dataset):
    prompt = example["conversations"][0]["content"]

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    baseline_results.append({
        "id": example["id"],
        "category": example["category"],
        "original": example["original"],
        "expected": example["optimized"],
        "model_output": answer,
    })

    print(f"{i+1}/40  {example['category']}  {example['id']}")

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1/40  no_change  no_change-003


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2/40  no_change  no_change-028


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

37/40  repeated_sort  repeated_sort-031


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


38/40  no_change  no_change-011


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


39/40  membership  membership-021
40/40  repeated_sort  repeated_sort-038


In [38]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]

    return {"text": texts}


train_formatted = dataset.map(
    formatting_prompts_func,
    batched=True,
)

val_formatted = val_dataset.map(
    formatting_prompts_func,
    batched=True,
)

print(train_formatted[0]["text"])

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

<|im_start|>user
Optimize this Python function while preserving its behavior.

Input constraints:
- chunks is a list of strings.

Return only the optimized Python function.

```python
def merge_words_2(chunks):
    out = ""
    for piece in chunks:
        out += piece
    return out
```<|im_end|>
<|im_start|>assistant
```python
def merge_words_2(chunks):
    return "".join(chunks)
```<|im_end|>



In [39]:
FastLanguageModel.for_training(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

In [40]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        dataset_text_field="text",

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        num_train_epochs=2,

        learning_rate=2e-4,
        warmup_ratio=0.1,

        logging_steps=1,

        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",

        seed=3407,
        report_to="none",

        output_dir="/kaggle/working/pyopt_checkpoints",
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/160 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/30 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [41]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<|im_start|>user\n' and response_part = '<|im_start|>assistant\n'


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [42]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 160 | Num Epochs = 2 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_avail

Step,Training Loss
1,0.757743
2,0.701004
3,1.165122
4,0.294568
5,0.541523
6,0.442073
7,0.431101
8,0.250645
9,0.124377
10,0.048838


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/pyopt_checkpoints/checkpoint-40/tokenizer_config.json.


In [43]:
save_path = "/kaggle/working/pyopt_lora"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved PyOpt LoRA to:", save_path)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/pyopt_lora/tokenizer_config.json.


Saved PyOpt LoRA to: /kaggle/working/pyopt_lora


In [44]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

In [45]:
import json
from pathlib import Path

pyopt_results = []

for i, example in enumerate(test_dataset):
    prompt = example["conversations"][0]["content"]

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    pyopt_results.append({
        "id": example["id"],
        "category": example["category"],
        "original": example["original"],
        "expected": example["optimized"],
        "model_output": answer,
    })

    print(f"{i+1}/40  {example['category']}  {example['id']}")

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1/40  no_change  no_change-003


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2/40  no_change  no_change-028


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

37/40  repeated_sort  repeated_sort-031


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


38/40  no_change  no_change-011


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


39/40  membership  membership-021
40/40  repeated_sort  repeated_sort-038


In [46]:
output_path = Path("/kaggle/working/pyopt_finetuned_outputs.jsonl")

with output_path.open("w") as f:
    for row in pyopt_results:
        f.write(json.dumps(row) + "\n")

print("Saved:", output_path)
print("Total results:", len(pyopt_results))

Saved: /kaggle/working/pyopt_finetuned_outputs.jsonl
Total results: 40


In [47]:
for i in range(5):
    print("=" * 80)
    print("ID:", pyopt_results[i]["id"])
    print("\nBASE:")
    print(baseline_results[i]["model_output"])
    print("\nFINETUNED:")
    print(pyopt_results[i]["model_output"])
    print("\nEXPECTED:")
    print(pyopt_results[i]["expected"])

ID: no_change-003

BASE:
```python
def map_squares_3(values):
    return [v ** 2 for v in values]
```

FINETUNED:
```python
def map_squares_3(values):
    return [entry * entry for entry in values]
```

EXPECTED:
def map_squares_3(values):
    return [entry * entry for entry in values]

ID: no_change-028

BASE:
```python
def map_squares_28(xs):
    return [x ** 2 for x in xs]
```

FINETUNED:
```python
def map_squares_28(xs):
    return [x * x for x in xs]
```

EXPECTED:
def map_squares_28(xs):
    return [x * x for x in xs]

ID: list_concat-003

BASE:
```python
def squares_3(items):
    return [number * number for number in items]
```

FINETUNED:
```python
def squares_3(items):
    output = []
    for number in items:
        output.append(number * number)
    return output
```

EXPECTED:
def squares_3(items):
    output = []
    for number in items:
        output.append(number * number)
    return output

ID: string_concat-021

BASE:
```python
def merge_words_21(parts):
    return ""

In [50]:
import re

def extract_code(text):
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

In [51]:
import timeit
import copy

def get_test_cases(category):
    if category == "membership":
        return [
            (list(range(100)), list(range(50, 150))),
            (list(range(1000)), list(range(500, 1500))),
        ]

    if category == "repeated_sum":
        return [
            (list(range(1, 101)),),
            (list(range(1, 1001)),),
        ]

    if category == "repeated_sort":
        return [
            (list(range(100, 0, -1)), 20),
            (list(range(1000, 0, -1)), 100),
        ]

    if category in {"list_concat", "no_change"}:
        return [
            (list(range(100)),),
            (list(range(1000)),),
        ]

    if category == "string_concat":
        return [
            (["abc"] * 100,),
            (["abc"] * 1000,),
        ]

    raise ValueError(category)

In [52]:
def evaluate_candidate(example, output_text):
    code = extract_code(output_text)

    try:
        namespace = {}
        exec(code, namespace)

        fn_name = example["fn"]

        if fn_name not in namespace:
            return {
                "valid": False,
                "correct": False,
                "speedup": None,
                "error": "Function name missing",
            }

        candidate = namespace[fn_name]

        original_ns = {}
        exec(example["original"], original_ns)
        original = original_ns[fn_name]

        cases = get_test_cases(example["category"])

        # Correctness
        for args in cases:
            expected = original(*copy.deepcopy(args))
            actual = candidate(*copy.deepcopy(args))

            if actual != expected:
                return {
                    "valid": True,
                    "correct": False,
                    "speedup": None,
                    "error": "Output mismatch",
                }

        # Benchmark on largest case
        args = cases[-1]

        original_time = timeit.timeit(
            lambda: original(*copy.deepcopy(args)),
            number=100,
        )

        candidate_time = timeit.timeit(
            lambda: candidate(*copy.deepcopy(args)),
            number=100,
        )

        return {
            "valid": True,
            "correct": True,
            "speedup": original_time / candidate_time,
            "error": None,
        }

    except Exception as e:
        return {
            "valid": False,
            "correct": False,
            "speedup": None,
            "error": str(e),
        }

In [53]:
comparison = []

for example, base, tuned in zip(
    test_dataset,
    baseline_results,
    pyopt_results,
):
    base_score = evaluate_candidate(
        example,
        base["model_output"],
    )

    tuned_score = evaluate_candidate(
        example,
        tuned["model_output"],
    )

    comparison.append({
        "id": example["id"],
        "category": example["category"],
        "base": base_score,
        "pyopt": tuned_score,
    })

print("Evaluated:", len(comparison))

Evaluated: 40


In [54]:
import statistics

def summarize(which):
    rows = [x[which] for x in comparison]

    valid = sum(x["valid"] for x in rows)
    correct = sum(x["correct"] for x in rows)

    speedups = [
        x["speedup"]
        for x in rows
        if x["correct"] and x["speedup"] is not None
    ]

    improved = sum(
        1 for x in speedups
        if x > 1.05
    )

    return {
        "valid_code": f"{valid}/40",
        "correct": f"{correct}/40",
        "meaningfully_faster": f"{improved}/{len(speedups)}",
        "median_speedup": statistics.median(speedups) if speedups else None,
    }

print("BASE QWEN")
print(summarize("base"))

print("\nPYOPT")
print(summarize("pyopt"))

BASE QWEN
{'valid_code': '40/40', 'correct': '40/40', 'meaningfully_faster': '29/40', 'median_speedup': 3.569178656723694}

PYOPT
{'valid_code': '40/40', 'correct': '40/40', 'meaningfully_faster': '36/40', 'median_speedup': 5.6533571951167545}


In [55]:
from collections import defaultdict
import statistics

categories = defaultdict(lambda: {
    "base_correct": 0,
    "pyopt_correct": 0,
    "base_speedups": [],
    "pyopt_speedups": [],
})

for row in comparison:
    cat = row["category"]

    if row["base"]["correct"]:
        categories[cat]["base_correct"] += 1
        categories[cat]["base_speedups"].append(
            row["base"]["speedup"]
        )

    if row["pyopt"]["correct"]:
        categories[cat]["pyopt_correct"] += 1
        categories[cat]["pyopt_speedups"].append(
            row["pyopt"]["speedup"]
        )

for cat, stats in categories.items():
    print("\n", cat)

    print(
        "Base median:",
        round(statistics.median(stats["base_speedups"]), 2)
    )

    print(
        "PyOpt median:",
        round(statistics.median(stats["pyopt_speedups"]), 2)
    )


 no_change
Base median: 0.98
PyOpt median: 1.0

 list_concat
Base median: 6.39
PyOpt median: 6.35

 string_concat
Base median: 1.26
PyOpt median: 1.27

 repeated_sort
Base median: 3.74
PyOpt median: 3.77

 membership
Base median: 1.01
PyOpt median: 11.58

 repeated_sum
Base median: 50.39
PyOpt median: 47.88


In [57]:
hard_tests = [
    {
        "id": "repeated_min_1",
        "category": "repeated_min",
        "fn": "shift_from_min",
        "original": """def shift_from_min(values):
    out = []
    for x in values:
        out.append(x - min(values))
    return out
""",
        "constraints": "values is a non-empty list of integers.",
    },
    {
        "id": "repeated_min_2",
        "category": "repeated_min",
        "fn": "distance_from_min",
        "original": """def distance_from_min(nums):
    result = []
    for n in nums:
        result.append(abs(n - min(nums)))
    return result
""",
        "constraints": "nums is a non-empty list of integers.",
    },
    {
        "id": "repeated_max_1",
        "category": "repeated_max",
        "fn": "normalize_by_max",
        "original": """def normalize_by_max(values):
    out = []
    for x in values:
        out.append(x / max(values))
    return out
""",
        "constraints": "values is a non-empty list of positive integers.",
    },
    {
        "id": "repeated_max_2",
        "category": "repeated_max",
        "fn": "gap_from_max",
        "original": """def gap_from_max(nums):
    result = []
    for x in nums:
        result.append(max(nums) - x)
    return result
""",
        "constraints": "nums is a non-empty list of integers.",
    },
    {
        "id": "repeated_dict_1",
        "category": "repeated_dict",
        "fn": "lookup_values",
        "original": """def lookup_values(keys, pairs):
    out = []
    for key in keys:
        if key in dict(pairs):
            out.append(dict(pairs)[key])
    return out
""",
        "constraints": "keys is a list of strings and pairs is a list of unique (string, integer) pairs.",
    },
    {
        "id": "repeated_dict_2",
        "category": "repeated_dict",
        "fn": "present_keys",
        "original": """def present_keys(keys, pairs):
    result = []
    for key in keys:
        if key in dict(pairs):
            result.append(key)
    return result
""",
        "constraints": "keys is a list of strings and pairs is a list of unique (string, integer) pairs.",
    },
    {
        "id": "nested_search_1",
        "category": "nested_search",
        "fn": "has_overlap",
        "original": """def has_overlap(a, b):
    for x in a:
        for y in b:
            if x == y:
                return True
    return False
""",
        "constraints": "a and b are lists of integers.",
    },
    {
        "id": "nested_search_2",
        "category": "nested_search",
        "fn": "common_exists",
        "original": """def common_exists(left, right):
    for x in left:
        for y in right:
            if x == y:
                return True
    return False
""",
        "constraints": "left and right are lists of integers.",
    },
    {
        "id": "repeated_tuple_1",
        "category": "repeated_tuple",
        "fn": "filter_allowed_tuple",
        "original": """def filter_allowed_tuple(items, allowed):
    out = []
    for x in items:
        if x in tuple(allowed):
            out.append(x)
    return out
""",
        "constraints": "items and allowed are lists of integers.",
    },
    {
        "id": "repeated_tuple_2",
        "category": "repeated_tuple",
        "fn": "keep_valid_tuple",
        "original": """def keep_valid_tuple(values, valid):
    result = []
    for x in values:
        if x in tuple(valid):
            result.append(x)
    return result
""",
        "constraints": "values and valid are lists of integers.",
    },
]

In [58]:
for example in hard_tests:
    example["prompt"] = f"""Optimize this Python function while preserving its behavior.

Input constraints:
- {example["constraints"]}

Return only the optimized Python function.

```python
{example["original"].strip()}
```"""

In [59]:
def generate_answer(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

In [60]:
hard_pyopt_results = []

for i, example in enumerate(hard_tests):
    answer = generate_answer(example["prompt"])

    hard_pyopt_results.append({
        **example,
        "model_output": answer,
    })

    print(f"PyOpt {i+1}/{len(hard_tests)}")

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PyOpt 1/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PyOpt 2/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

PyOpt 7/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PyOpt 8/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PyOpt 9/10
PyOpt 10/10


In [61]:
hard_base_results = []

with model.disable_adapter():
    for i, example in enumerate(hard_tests):
        answer = generate_answer(example["prompt"])

        hard_base_results.append({
            **example,
            "model_output": answer,
        })

        print(f"Base {i+1}/{len(hard_tests)}")

Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 1/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 2/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Base 7/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 8/10


Both `max_new_tokens` (=300) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 9/10
Base 10/10


In [62]:
for i in range(len(hard_tests)):
    print("=" * 80)
    print(hard_tests[i]["id"])

    print("\nBASE:")
    print(hard_base_results[i]["model_output"])

    print("\nPYOPT:")
    print(hard_pyopt_results[i]["model_output"])

repeated_min_1

BASE:
```python
def shift_from_min(values):
    min_val = min(values)
    return [x - min_val for x in values]
```

PYOPT:
```python
def shift_from_min(values):
    min_value = min(values)
    out = []
    for x in values:
        out.append(x - min_value)
    return out
```
repeated_min_2

BASE:
```python
def distance_from_min(nums):
    min_val = min(nums)
    return [abs(n - min_val) for n in nums]
```

PYOPT:
```python
def distance_from_min(nums):
    min_value = min(nums)
    result = []
    for n in nums:
        result.append(abs(n - min_value))
    return result
```
repeated_max_1

BASE:
```python
def normalize_by_max(values):
    max_val = max(values)
    return [x / max_val for x in values]
```

PYOPT:
```python
def normalize_by_max(values):
    max_value = max(values)
    out = []
    for x in values:
        out.append(x / max_value)
    return out
```
repeated_max_2

BASE:
```python
def gap_from_max(nums):
    max_val = max(nums)
    return [max_val - x for

In [64]:
def hard_test_cases(example_id):
    cases = {
        "repeated_min_1": [
            ([3, 1, 5],),
            ([10],),
            ([-2, 4, 0],),
        ],
        "repeated_min_2": [
            ([3, 1, 5],),
            ([10],),
            ([-2, 4, 0],),
        ],
        "repeated_max_1": [
            ([1, 2, 4],),
            ([5],),
            ([3, 7, 2],),
        ],
        "repeated_max_2": [
            ([1, 2, 4],),
            ([5],),
            ([-3, 7, 2],),
        ],
        "repeated_dict_1": [
            (["a", "b"], [("a", 1), ("b", 2)]),
            (["b", "a"], [("a", 1), ("b", 2)]),
            (["a", "a"], [("a", 1), ("b", 2)]),
            (["x"], [("a", 1), ("b", 2)]),
        ],
        "repeated_dict_2": [
            (["a", "b"], [("a", 1), ("b", 2)]),
            (["b", "a"], [("a", 1), ("b", 2)]),
            (["a", "a"], [("a", 1), ("b", 2)]),
            (["x"], [("a", 1), ("b", 2)]),
        ],
        "nested_search_1": [
            ([1, 2], [3, 4]),
            ([1, 2], [2, 4]),
            ([1, 1], [3, 4]),
            ([], [1, 2]),
        ],
        "nested_search_2": [
            ([1, 2], [3, 4]),
            ([1, 2], [2, 4]),
            ([1, 1], [3, 4]),
            ([], [1, 2]),
        ],
        "repeated_tuple_1": [
            ([1, 2, 3], [2, 3]),
            ([1, 1, 2], [1]),
            ([], [1]),
            ([1, 2], []),
        ],
        "repeated_tuple_2": [
            ([1, 2, 3], [2, 3]),
            ([1, 1, 2], [1]),
            ([], [1]),
            ([1, 2], []),
        ],
    }

    return cases[example_id]

In [67]:
import copy
import timeit

def evaluate_hard(example, output_text):
    code = extract_code(output_text)

    try:
        original_ns = {}
        exec(example["original"], original_ns)
        original_fn = original_ns[example["fn"]]

        candidate_ns = {}
        exec(code, candidate_ns)

        if example["fn"] not in candidate_ns:
            return {
                "correct": False,
                "speedup": None,
                "error": "Function name missing"
            }

        candidate_fn = candidate_ns[example["fn"]]

        cases = hard_test_cases(example["id"])

        for args in cases:
            expected = original_fn(*copy.deepcopy(args))
            actual = candidate_fn(*copy.deepcopy(args))

            if actual != expected:
                return {
                    "correct": False,
                    "speedup": None,
                    "error": f"Mismatch on {args}: {actual!r} != {expected!r}"
                }

        # benchmark largest useful case separately
        return {
            "correct": True,
            "speedup": None,
            "error": None
        }

    except Exception as e:
        return {
            "correct": False,
            "speedup": None,
            "error": str(e)
        }

In [70]:
hard_comparison = []

for example, base, tuned in zip(
    hard_tests,
    hard_base_results,
    hard_pyopt_results,
):
    base_score = evaluate_hard(
        example,
        base["model_output"]
    )

    pyopt_score = evaluate_hard(
        example,
        tuned["model_output"]
    )

    hard_comparison.append({
        "id": example["id"],
        "base": base_score,
        "pyopt": pyopt_score,
    })

for row in hard_comparison:
    print(row["id"])
    print("  BASE :", row["base"])
    print("  PYOPT:", row["pyopt"])

repeated_min_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_min_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_max_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_max_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_dict_1
  BASE : {'correct': False, 'speedup': None, 'error': "Mismatch on (['b', 'a'], [('a', 1), ('b', 2)]): [1, 2] != [2, 1]"}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
repeated_dict_2
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': True, 'speedup': None, 'error': None}
nested_search_1
  BASE : {'correct': True, 'speedup': None, 'error': None}
  PYOPT: {'correct': False, 'speedup': None, 'error': 'Mismatc

In [71]:
extra_examples = [
    {
        "conversations": [
            {
                "role": "user",
                "content": """Optimize this Python function while preserving its behavior.

Input constraints:
- left and right are lists of integers.

Return only the optimized Python function.

```python
def overlaps(left, right):
    for x in left:
        for y in right:
            if x == y:
                return True
    return False
```"""
            },
            {
                "role": "assistant",
                "content": """```python
def overlaps(left, right):
    right_lookup = set(right)
    for x in left:
        if x in right_lookup:
            return True
    return False
```"""
            },
        ]
    },

    {
        "conversations": [
            {
                "role": "user",
                "content": """Optimize this Python function while preserving its behavior.

Input constraints:
- values and allowed are lists of integers.

Return only the optimized Python function.

```python
def keep_allowed(values, allowed):
    result = []
    for x in values:
        if x in tuple(allowed):
            result.append(x)
    return result
```"""
            },
            {
                "role": "assistant",
                "content": """```python
def keep_allowed(values, allowed):
    allowed_lookup = set(allowed)
    result = []
    for x in values:
        if x in allowed_lookup:
            result.append(x)
    return result
```"""
            },
        ]
    },

    {
        "conversations": [
            {
                "role": "user",
                "content": """Optimize this Python function while preserving its behavior.

Input constraints:
- keys is a list of strings.
- pairs contains unique string keys and integer values.
- output order and duplicate keys from keys must be preserved.

Return only the optimized Python function.

```python
def fetch(keys, pairs):
    out = []
    for key in keys:
        if key in dict(pairs):
            out.append(dict(pairs)[key])
    return out
```"""
            },
            {
                "role": "assistant",
                "content": """```python
def fetch(keys, pairs):
    lookup = dict(pairs)
    out = []
    for key in keys:
        if key in lookup:
            out.append(lookup[key])
    return out
```"""
            },
        ]
    },
]